## Importing the libraries

In [1]:
import pandas as pd

from src.utils import show_map
from src.links import HeliumLink
from src.collect import *
from os.path import isfile

%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False

pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.max_colwidth', 25)

First of all, run the following cell to initialize the Google Earth API. The output will contain instructions on how to grant this notebook access to Earth Engine using your account.

In [3]:
import ee

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize()

# Import the USGS ground elevation image.
elv = ee.Image('USGS/SRTMGL1_003')


Successfully saved authorization token.


## Getting data  
This code collects the information of nodes that are inside 50 km around the specified point.

In [4]:
# Interest area parameters
main_lat = 43.9212891
main_lon = -78.8100379
coverage_area = 50000

# Helium nodes on Courtice, Ontario CA with altitude from 
# Google Earth Engine API at March 9, 2022 
path = './databases/Courtice_nodes.csv'
if isfile(path):
    print("The DataFrame was previously loaded.")
    df = pd.read_csv(path)
else:
    # If there is not the file, the code uses the Open Topo Data
    # for get the altitudes of all nodes
    print("Getting nodes...")
    df = pd.DataFrame(get_nodes(main_lat, main_lon, coverage_area)) 
    df.to_csv(path, index=False)
df.head(5).T    

The DataFrame was previously loaded.


,0,1,2,3,4
name,acidic-cyan-python,docile-sky-millipede,cheerful-peach-pheasant,shaggy-cinnamon-falcon,tall-tartan-kestrel
longitude,-78.800,-78.821,-78.799,-78.821,-78.820
latitude,43.920,43.922,43.921,43.922,43.927
distance,825.506,871.338,880.732,889.502,1008.547
gain,40,58,80,12,40
elevation,0,3,8,0,2
reward_scale,1.000,0.500,1.000,0.500,1.000
address,112JNUn5JsfnuAvdXD1be...,11xowkkf7wPTjqAEgX7sh...,118svoyi7Es5QFBjNcNT8...,112gdYHwTi8n6wAhZeUz3...,112sUtNsafzbAavGYTk1E...
altitude,140,147,145,147,147


The important information is saved in a dataframe.

## Showing nodes in the map  
The following map shows the location of all nodes that are within 50 km of the specified point (lat, lon). The n parameter is used to display the n furthest nodes.

In [5]:
show_map(df, main_lat, main_lon, n=600)

## Checking the links

In [6]:
# The reference map matches all GPS points to a matrix os 1100x1100 
# in which each cell have a longitude of 150 m (11000/100 = 1100).
# This map is used to chek the Fresnel zone.
cell_parameters = {
    "cells_in_x": 1100,
    "cells_in_y": 1100,
    "initial_cell_x": 550,
    "initial_cell_y": 550,
    "map_width_in_mtrs": 110_000,
    "map_long_in_mtrs": 110_000,
    "initial_coordinate": (main_lat, main_lon),
}

# Creating the main object
links = HeliumLink(df, cell_parameters)

In [7]:
links.set_location_type('rural')

The location type is rural


In [8]:
# The random test_sample is used only to get the latitude and longitude 
# of a gateway in the network
test_sample = df.sample(1).iloc[0]

simulated_point_parameters = {
    'latitude' : test_sample.latitude,
    'longitude' : test_sample.longitude,
    'gains' : range(1,11),
    'elevations' : range(5,16),
    'Google_Earth_Auntentificator' : ee,  
    'Google_Earth_Elevation_Image' : elv,
}

resp = links.check(**simulated_point_parameters)
resp

gain 5, elevaion 1
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code cursor 1 = 200
gain 5, elevaion 2
Getting witnesses for a Hotspot - status_code cursor 1 = 200
Getting witnesses for a Hotspot - status_code c

,gain,elevation,earings,nodes
0,5,1,49.372,[original-orange-snak...
1,5,2,49.372,[alert-sandstone-croc...
2,5,3,52.971,[alert-sandstone-croc...
3,5,4,52.971,[alert-sandstone-croc...
4,5,5,52.971,[alert-sandstone-croc...
...,...,...,...,...
105,15,6,78.704,[alert-sandstone-croc...
106,15,7,105.781,[alert-sandstone-croc...
107,15,8,108.904,[alert-sandstone-croc...
108,15,9,117.171,[alert-sandstone-croc...


In [17]:
# Saving the final responce in the databases folder
path = './databases/final_response.csv'
resp.to_csv(path, index=False)